# RIDI in 60 seconds

**Question:** can a system look almost unchanged globally while the finite identities receiving action change completely?

This notebook is deterministic and uses synthetic data. It demonstrates the tool; it is **not manuscript evidence**.

In [ ]:
%pip -q install git+https://github.com/adeebnoor/ridi.git

## 1. The lightest possible audit

If your pipeline already exposes the selected identities, one function is enough.

In [ ]:
from ridi_audit import compare_allocations

reference = ['doc-1', 'doc-2', 'doc-3', 'doc-4']
updated   = ['doc-1', 'doc-2', 'doc-9', 'doc-4']

report = compare_allocations(reference, updated)
print(report)

## 2. Why global rank agreement is not enough

Now create two rankings that differ only in two adjacent blocks of size `k`. As the candidate universe grows, global Spearman agreement approaches 1 while the entire top-*k* allocation is replaced.

In [ ]:
n = 10_000
k = 50
assert 1 <= k <= n // 2

r0 = list(range(n))
r1 = r0[k:2*k] + r0[:k] + r0[2*k:]
rho = 1 - 12*k**3 / (n*(n**2 - 1))
allocation = compare_allocations(r0[:k], r1[:k])

print(f'Candidates:                 {n:,}')
print(f'Decision capacity:          {k:,}')
print(f'Global Spearman agreement:  {rho:.9f}')
print(f'Top-k overlap:              {allocation.overlap}/{k}')
print(f'RIDI:                       {allocation.ridi:.3f}')

For this construction, global rank agreement tends to 1 as `n` increases while `RIDI=1`. Global agreement therefore cannot certify finite allocation identity.

## 3. Score-table audit and control

When you have paired candidate scores, `audit()` adds deterministic top-k selection, changed slots, global rank agreement, a sufficient score-margin stability certificate, and access to the identity–utility frontier.

In [ ]:
import pandas as pd
from ridi_audit import audit

before = pd.DataFrame({
    'id': ['a','b','c','d','e','f'],
    'score': [0.99,0.94,0.90,0.85,0.81,0.76],
})
after = pd.DataFrame({
    'id': ['a','b','c','d','e','f'],
    'score': [0.98,0.93,0.72,0.86,0.80,0.89],
})

score_report = audit(before, after, k=[3,5])
print(score_report)

## Use it in your research

RIDI is framework-agnostic: RAG contexts, vulnerability queues, clinical alerts, fraud review, hiring shortlists, grant review, inspection queues, and any other finite score-to-action pipeline.

It measures **who changed**. It does not by itself establish harm, fairness, correctness or model superiority.

Next: [Quick Start](https://github.com/adeebnoor/ridi/blob/main/docs/QUICKSTART.md) · [Use cases](https://github.com/adeebnoor/ridi/blob/main/docs/USE_CASES.md) · [Reporting checklist](https://github.com/adeebnoor/ridi/blob/main/docs/REPORTING_CHECKLIST.md)